# 8.2 剪枝 (Pruning)

剪枝通过移除不重要的参数来减少模型大小和计算量。

本节涵盖：
- 非结构化剪枝
- 结构化剪枝
- LLM剪枝

## 1. 非结构化剪枝

**目的**：移除不重要的单个权重

**基本原理**：将绝对值小于阈值的权重置零（创建稀疏矩阵），按权重重要性排序，移除最不重要的权重。

**核心方法**：幅度剪枝（Magnitude Pruning）
- 排序权重绝对值
- 移除最小的p%权重
- 产生非结构化稀疏（零值分散在矩阵中）

**局限**：非结构化稀疏需要专门的稀疏硬件/软件支持才能加速

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

def magnitude_prune(tensor, sparsity=0.5):
    abs_tensor = tensor.abs()
    threshold = torch.quantile(abs_tensor.flatten(), sparsity)
    mask = (abs_tensor > threshold).float()
    pruned = tensor * mask
    return pruned, mask

W = torch.randn(256, 256)
W_pruned, mask = magnitude_prune(W, sparsity=0.5)

print('=== Unstructured Pruning ===')
print(f'Original non-zero: {(W != 0).sum().item():,} / {W.numel():,}')
print(f'After 50% pruning non-zero: {(W_pruned != 0).sum().item():,} / {W.numel():,}')
print(f'Actual sparsity: {(W_pruned == 0).float().mean():.2%}')

for sparsity in [0.3, 0.5, 0.7, 0.9]:
    W_p, m = magnitude_prune(W, sparsity)
    error = (W - W_p).norm() / W.norm()
    print(f'  Sparsity {sparsity:.0%}: relative error = {error:.4f}')

print(f'\nKey: Unstructured pruning creates sparse matrices.')
print(f'Needs specialized sparse kernels for actual speedup.')

## 2. 结构化剪枝

**目的**：移除整个神经元/注意力头/层

**基本原理**：按结构单元（通道、注意力头、层）评估重要性，移除整个结构单元，产生规则的小矩阵，无需稀疏支持即可加速。

**结构化剪枝类型**：
- **通道剪枝**：移除整个输出通道
- **头剪枝**：移除整个注意力头
- **层剪枝**：移除整个Transformer层

**优势**：直接减少矩阵维度，任何硬件都能加速

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class StructuredPrunedModel(nn.Module):
    def __init__(self, d_model=256, d_ff=1024, n_heads=8, prune_heads=3):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.keep_heads = n_heads - prune_heads
        self.keep_dim = self.keep_heads * self.d_head

        self.qkv = nn.Linear(d_model, 3 * self.keep_dim)
        self.proj = nn.Linear(self.keep_dim, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.keep_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, self.keep_dim)
        return self.proj(out) + self.ffn(x)

full_model = nn.TransformerEncoderLayer(256, 8, 1024, batch_first=True)
pruned_model = StructuredPrunedModel(256, 1024, 8, prune_heads=3)

full_params = sum(p.numel() for p in full_model.parameters())
pruned_params = sum(p.numel() for p in pruned_model.parameters())

x = torch.randn(2, 16, 256)
out = pruned_model(x)

print('=== Structured Pruning ===')
print(f'Full model: {full_params:,} params (8 heads)')
print(f'Pruned model: {pruned_params:,} params (5 heads, 3 pruned)')
print(f'Parameter reduction: {(1 - pruned_params/full_params):.1%}')
print(f'Output shape: {out.shape}')
print(f'\nKey: Structured pruning removes entire heads/channels.')
print(f'Directly reduces matrix dimensions -> actual speedup on any hardware.')

## 3. LLM剪枝

**目的**：专门针对大语言模型的剪枝

**代表方法**：
- **ShortGPT**：移除冗余的Transformer层，发现深层中存在大量冗余
- **SliceGPT**：通过奇异值分解移除主成分的次要分量
- **LLM-Pruner**：基于依赖关系的结构化剪枝

**LLM剪枝的挑战**：
- 剪枝后需要少量微调恢复性能
- 剪枝比例受限于模型能力下降
- 通常20-30%剪枝率后性能显著下降

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class ShortGPTDemo(nn.Module):
    def __init__(self, d_model=128, n_heads=2, n_layers=6, keep_layers=4):
        super().__init__()
        all_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, batch_first=True)
            for _ in range(n_layers)
        ])
        self.layers = nn.ModuleList([all_layers[i] for i in range(keep_layers)])
        self.n_original = n_layers
        self.n_kept = keep_layers

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

def layer_redundancy_score(model, x, layer_idx):
    h = x.clone()
    for i in range(layer_idx):
        h = model.layers[i](h)
    h_before = h.clone()
    h_after = model.layers[layer_idx](h_before)
    similarity = torch.cosine_similarity(h_before.flatten(), h_after.flatten(), dim=0)
    return similarity.item()

full_model = ShortGPTDemo(128, 2, 6, 6)
pruned_model = ShortGPTDemo(128, 2, 6, 4)
x = torch.randn(2, 16, 128)

full_params = sum(p.numel() for p in full_model.parameters())
pruned_params = sum(p.numel() for p in pruned_model.parameters())

print('=== LLM Pruning (ShortGPT-style) ===')
print(f'Original: {full_model.n_original} layers, {full_params:,} params')
print(f'Pruned: {pruned_model.n_kept} layers, {pruned_params:,} params')
print(f'Layer reduction: {(1 - pruned_params/full_params):.1%}')

print(f'\nLayer redundancy scores (cosine similarity):')
for i in range(min(4, len(full_model.layers))):
    score = layer_redundancy_score(full_model, x, i)
    print(f'  Layer {i}: {score:.4f} (higher = more redundant)')
print(f'\nKey: Remove layers with highest redundancy scores.')
print(f'Usually need LoRA fine-tuning after pruning to recover performance.')